# Lab: Data Collection dari Twitter/X menggunakan TwitterAPI.io

**Matakuliah:** Social Media Analytics  
**Tools:** Python, TwitterAPI.io, NetworkX, Pandas  

---

### Tujuan Lab
Setelah menyelesaikan lab ini, Anda akan mampu:
1. Memahami mengapa third-party API diperlukan dan bagaimana cara kerjanya
2. Menggunakan TwitterAPI.io dari Python untuk mengumpulkan tweet berdasarkan keyword/hashtag
3. Mengumpulkan data replies untuk membangun conversation network
4. Menyimpan dan memuat ulang data yang sudah dikumpulkan
5. Menghubungkan data Twitter dengan analisis network yang sudah dipelajari di Lab 1-4

### Cara Menggunakan Notebook Ini
- **Jalankan setiap cell secara berurutan** (Shift+Enter)
- Cell bertanda **[Pertanyaan]** berisi pertanyaan -- jawab di cell yang disediakan
- Cell bertanda **[Latihan]** berisi latihan koding -- tulis kode Anda di sana
- Bagian terakhir adalah **Tugas** yang harus dikumpulkan

> **Penting:** API key TwitterAPI.io sudah disediakan oleh dosen. Jangan bagikan key ini ke luar kelas.

---
## 0. Persiapan

Kita mulai dengan mengimpor semua library yang dibutuhkan. TwitterAPI.io menggunakan library `requests` standar Python -- tidak ada library khusus yang perlu diinstal.

In [ ]:
# Semua library yang dibutuhkan sudah tersedia di Google Colab
import json
import time
import requests
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

# Pengaturan tampilan
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_colwidth', 80)

print(f"Versi NetworkX : {nx.__version__}")
print(f"Versi Pandas   : {pd.__version__}")
print("Persiapan selesai.")

In [ ]:
# ============================================================
#  MASUKKAN API KEY DI SINI
#  Key dari dosen -- jangan share ke luar kelas!
# ============================================================
TWITTERAPI_KEY = "new1_954d010bf9e847f09c8b77f4b34bb3cb"

# Header yang akan digunakan di setiap request
HEADERS  = {"X-API-Key": TWITTERAPI_KEY}
BASE_URL = "https://api.twitterapi.io"

# Verifikasi koneksi ke API
try:
    resp = requests.get(
        f"{BASE_URL}/twitter/tweet/advanced_search",
        headers=HEADERS,
        params={"query": "test", "queryType": "Latest"},
        timeout=10
    )
    if resp.status_code == 200:
        print("Koneksi ke TwitterAPI.io berhasil.")
    elif resp.status_code == 401:
        print("API key tidak valid. Pastikan Anda sudah mengisi TWITTERAPI_KEY dengan benar.")
    else:
        print(f"Status: {resp.status_code} -- {resp.text[:100]}")
except Exception as e:
    print(f"Gagal terhubung: {e}")

---
## 1. Mengapa Kita Menggunakan Third-Party API?

Sebelum masuk ke kode, mari kita pahami konteksnya.

Twitter/X dulunya menyediakan **API gratis** yang memungkinkan peneliti mengambil tweet secara langsung. Sejak 2023, akses API resmi menjadi **berbayar** -- paket termurahnya $100/bulan hanya untuk 10.000 tweet per bulan.

Alternatif yang digunakan di lab ini adalah **TwitterAPI.io**, sebuah layanan pihak ketiga yang mengakses data Twitter secara tidak langsung. Harganya jauh lebih terjangkau ($0.15 per 1.000 tweet) dan tidak membutuhkan akun Twitter developer.

```
Kode Python Anda
      |
      v
 TwitterAPI.io  ------>  Twitter/X
      |
      v
  Data Tweet (JSON)
```

Endpoint yang akan kita gunakan:

| Endpoint | Kegunaan |
|---|---|
| `/twitter/tweet/advanced_search` | Cari tweet berdasarkan keyword/hashtag |
| `/twitter/tweet/replies` | Ambil replies dari sebuah tweet |
| `/twitter/tweet/quotes` | Ambil quote tweets dari sebuah tweet |

Ketiganya dipanggil dengan cara yang sama: HTTP GET request dengan API key di header.

---
## 2. Mencari Tweet Berdasarkan Keyword

Kita akan mulai dengan kasus paling umum: **mencari tweet yang membahas topik tertentu**.

TwitterAPI.io mendukung **Twitter advanced search syntax** -- bahasa query yang sama dengan yang bisa Anda ketik di kolom pencarian Twitter. Contoh:

| Query | Artinya |
|---|---|
| `banjir jakarta lang:id` | Tweet mengandung kata *banjir* DAN *jakarta*, Bahasa Indonesia |
| `#pilkada2024 lang:id` | Tweet dengan hashtag #pilkada2024, Bahasa Indonesia |
| `"harga beras" since:2024-01-01 lang:id` | Frasa tepat sejak tanggal tertentu |
| `from:jokowi` | Tweet dari akun tertentu |
| `banjir -is:retweet lang:id` | Tweet tentang banjir, tidak termasuk retweet |

Menambahkan `lang:id` langsung di query lebih reliable daripada mengandalkan filter bahasa di parameter API, karena filter ini diproses langsung oleh mesin pencari Twitter.

In [ ]:
# ==============================================================
#  FUNGSI UTAMA: Cari tweet berdasarkan keyword
# ==============================================================

def cari_tweet(query, maks_tweet=100, simpan_ke=None):
    """
    Mencari tweet berdasarkan query menggunakan TwitterAPI.io.

    API ini mengembalikan maksimum 20 tweet per halaman.
    Fungsi ini otomatis melakukan pagination sampai maks_tweet tercapai.

    Parameter:
        query      : string pencarian (mendukung Twitter advanced search syntax)
        maks_tweet : jumlah maksimum tweet yang diambil
        simpan_ke  : (opsional) path file JSON untuk menyimpan hasil

    Return:
        list of dict -- setiap dict adalah satu tweet
    """
    print(f"Mencari tweet: '{query}'")
    print(f"Maksimum: {maks_tweet} tweet")

    semua_tweet  = []
    ids_seen     = set()
    cursor       = None
    cursor_lama  = None
    halaman      = 1

    while len(semua_tweet) < maks_tweet:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        resp = requests.get(
            f"{BASE_URL}/twitter/tweet/advanced_search",
            headers=HEADERS, params=params, timeout=15
        )

        if resp.status_code != 200:
            print(f"Error {resp.status_code}: {resp.text[:200]}")
            break

        data = resp.json()
        tweets_halaman = data.get("tweets", [])

        if not tweets_halaman:
            print("Tidak ada tweet lagi.")
            break

        baru = [t for t in tweets_halaman if t.get('id') not in ids_seen]
        if not baru:
            print("API mengembalikan halaman yang sama -- berhenti.")
            break
        for t in baru:
            ids_seen.add(t.get('id'))
        semua_tweet.extend(baru)
        print(f"  Halaman {halaman}: {len(baru)} tweet baru (total: {len(semua_tweet)})")

        cursor_baru = data.get("next_cursor")
        if not cursor_baru or cursor_baru == cursor_lama:
            break
        cursor_lama = cursor
        cursor      = cursor_baru

        halaman += 1
        time.sleep(0.5)

    semua_tweet = semua_tweet[:maks_tweet]
    print(f"Selesai. Total: {len(semua_tweet)} tweet.")

    if simpan_ke:
        with open(simpan_ke, 'w', encoding='utf-8') as f:
            json.dump(semua_tweet, f, ensure_ascii=False, indent=2)
        print(f"Data disimpan ke: {simpan_ke}")

    return semua_tweet

print("Fungsi cari_tweet() siap digunakan.")

In [ ]:
# Jalankan pencarian.
# dan simpan hasilnya agar tidak perlu memanggil API lagi di sesi berikutnya.

QUERY = "Iran -is:retweet min_replies:3"   # <-- ganti keyword di sini

tweets = cari_tweet(
    query=QUERY,
    maks_tweet=150,
    simpan_ke="tweets_hasil.json"
)

### 2.1 Melihat Struktur Data

Sebelum menganalisis, kita perlu memahami **struktur data** yang dikembalikan. Satu tweet direpresentasikan sebagai sebuah dictionary Python dengan banyak field.

In [ ]:
# Lihat semua field yang tersedia pada tweet pertama
tweet_contoh = tweets[0]

print("=" * 60)
print("FIELD-FIELD DALAM SATU TWEET:")
print("=" * 60)
for key, value in tweet_contoh.items():
    nilai_str = str(value)
    if len(nilai_str) > 60:
        nilai_str = nilai_str[:60] + "..."
    print(f"  {key:<25} : {nilai_str}")

In [ ]:
# Field-field penting untuk analisis:
print("FIELD PENTING:")
print("-" * 60)
print(f"  ID tweet         : {tweet_contoh.get('id')}")
print(f"  Teks             : {tweet_contoh.get('text', '')}")
print(f"  Waktu            : {tweet_contoh.get('createdAt')}")
print(f"  Username         : @{tweet_contoh.get('author', {}).get('userName', 'N/A')}")
print(f"  Nama lengkap     : {tweet_contoh.get('author', {}).get('name', 'N/A')}")
print(f"  Jumlah like      : {tweet_contoh.get('likeCount', 0)}")
print(f"  Jumlah retweet   : {tweet_contoh.get('retweetCount', 0)}")
print(f"  Jumlah reply     : {tweet_contoh.get('replyCount', 0)}")
print(f"  Jumlah quote     : {tweet_contoh.get('quoteCount', 0)}")
print(f"  isReply          : {tweet_contoh.get('isReply', False)}")
print(f"  inReplyToId      : {tweet_contoh.get('inReplyToId', None)}")
print(f"  inReplyToUsername: {tweet_contoh.get('inReplyToUsername', None)}")
print(f"  conversationId   : {tweet_contoh.get('conversationId', None)}")
print()
print("Perhatikan: field inReplyToUsername tersedia secara langsung,")
print("sehingga kita tidak perlu lookup tambahan saat membangun network.")

### 2.2 Konversi ke DataFrame

Untuk analisis yang lebih mudah, kita konversi data ke **Pandas DataFrame**.

In [ ]:
def tweets_ke_dataframe(tweets):
    """
    Mengekstrak field-field penting dari list tweet mentah
    dan mengembalikannya sebagai Pandas DataFrame.
    """
    baris = []
    for t in tweets:
        author = t.get('author', {})
        baris.append({
            'tweet_id'            : t.get('id'),
            'teks'                : t.get('text', ''),
            'waktu'               : t.get('createdAt'),
            'username'            : author.get('userName', ''),
            'nama'                : author.get('name', ''),
            'followers'           : author.get('followers', 0),
            'like'                : t.get('likeCount', 0),
            'retweet'             : t.get('retweetCount', 0),
            'reply'               : t.get('replyCount', 0),
            'quote'               : t.get('quoteCount', 0),
            'is_reply'            : t.get('isReply', False),
            'in_reply_to_id'      : t.get('inReplyToId'),
            'in_reply_to_username': t.get('inReplyToUsername'),
            'conversation_id'     : t.get('conversationId'),
        })
    df = pd.DataFrame(baris)
    df['waktu'] = pd.to_datetime(df['waktu'])
    return df

df = tweets_ke_dataframe(tweets)

print(f"Shape DataFrame: {df.shape} (baris, kolom)")
print(f"\nContoh 5 tweet pertama:")
df[['username', 'teks', 'like', 'retweet', 'reply']].head()

In [ ]:
# Statistik dasar engagement
print("STATISTIK ENGAGEMENT:")
print("-" * 40)
print(df[['like', 'retweet', 'reply', 'quote']].describe().round(1))

print("\n10 TWEET DENGAN ENGAGEMENT TERTINGGI:")
df['total_engagement'] = df['like'] + df['retweet'] + df['reply']
df.sort_values('total_engagement', ascending=False)[['username', 'teks', 'total_engagement']].head(10)

### 2.3 Distribusi Aktivitas Pengguna

Salah satu pertanyaan paling mendasar dalam analisis media sosial: **apakah semua pengguna sama aktifnya?**

In [ ]:
tweet_per_user = df['username'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(tweet_per_user.values, bins=range(1, tweet_per_user.max()+2),
             color='#065A82', edgecolor='white', alpha=0.85, align='left')
axes[0].set_xlabel('Jumlah tweet dalam dataset', fontsize=11)
axes[0].set_ylabel('Jumlah pengguna', fontsize=11)
axes[0].set_title('Distribusi Aktivitas Pengguna\n(Berapa tweet yang dikirim tiap orang?)',
                  fontsize=12, fontweight='bold')
axes[0].axvline(tweet_per_user.mean(), color='red', linestyle='--',
               label=f'Rata-rata = {tweet_per_user.mean():.1f}')
axes[0].legend()

top10 = tweet_per_user.head(10)
axes[1].barh(top10.index[::-1], top10.values[::-1], color='#3498DB', alpha=0.85)
axes[1].set_xlabel('Jumlah tweet', fontsize=11)
axes[1].set_title('10 Pengguna Paling Aktif', fontsize=12, fontweight='bold')
for i, (val, name) in enumerate(zip(top10.values[::-1], top10.index[::-1])):
    axes[1].text(val + 0.1, i, str(val), va='center', fontsize=10)

plt.tight_layout()
plt.show()

persen_satu_tweet = (tweet_per_user == 1).sum() / len(tweet_per_user) * 100
print(f"Pengguna yang hanya tweet 1x: {persen_satu_tweet:.1f}%")
print("Pola ini disebut distribusi 'long-tail', umum ditemukan di media sosial.")

### [Pertanyaan 1]

Perhatikan histogram distribusi aktivitas pengguna di atas.

1. Apakah distribusi aktivitas pengguna terlihat seperti **bell curve** (normal) atau memiliki **ekor panjang** ke kanan?
2. Ingat distribusi degree pada jaringan Barabasi-Albert dari Lab 1. Apakah pola aktivitas pengguna Twitter ini mirip dengan model network tersebut? Mengapa?
3. Jika kita membuat network mention dari data ini, node mana yang kemungkinan besar akan menjadi **hub** (simpul dengan banyak koneksi)?

**Jawaban Anda:**

1. ...
2. ...
3. ...

---
## 3. Memuat Ulang Data yang Sudah Disimpan

Setiap pemanggilan API menghabiskan kredit dan waktu. Gunakan data yang sudah tersimpan jika tidak ada perubahan yang diperlukan -- ini praktik yang **wajib dilakukan** dalam riset.

In [ ]:
def muat_tweets(path_file):
    """
    Memuat tweet dari file JSON yang sudah disimpan sebelumnya.
    Gunakan fungsi ini agar tidak perlu memanggil API lagi.
    """
    with open(path_file, 'r', encoding='utf-8') as f:
        tweets = json.load(f)
    print(f"Berhasil memuat {len(tweets)} tweet dari '{path_file}'")
    return tweets

# Contoh penggunaan -- di sesi berikutnya, cukup jalankan dua baris ini:
# tweets = muat_tweets("tweets_hasil.json")
# df     = tweets_ke_dataframe(tweets)

print("Fungsi muat_tweets() siap digunakan.")
print()
print("Tips: Di sesi lab berikutnya, gunakan muat_tweets() daripada")
print("      memanggil cari_tweet() lagi, untuk menghemat kredit API.")

---
## 4. Membangun Conversation Network dari Replies

Di Lab 1-3, kita sudah belajar menganalisis network yang sudah jadi. Sekarang kita akan **membangun network dari data real**.

### Ide Dasarnya

Setiap kali seseorang **membalas (reply)** tweet orang lain, terjadi sebuah interaksi terarah:

```
@budi  ----reply---->  @siti
```

Jika kita kumpulkan semua interaksi ini, kita bisa membangun sebuah **directed network** di mana:
- **Node** = pengguna Twitter
- **Edge** = satu user membalas tweet user lain
- **Bobot edge** = berapa kali mereka berinteraksi

### Strategi Pengumpulan Data

Kita tidak bisa mengambil semua reply di Twitter sekaligus. Strateginya:
1. Cari tweet populer tentang topik yang dipilih (sudah dilakukan di Bagian 2)
2. Pilih tweet yang memiliki banyak reply
3. Ambil semua reply dari tweet-tweet tersebut
4. Bangun network dari hubungan reply tersebut

In [ ]:
# Langkah 1: Pilih tweet dengan banyak reply sebagai titik awal network
df_kandidat = df[df['reply'] > 5].sort_values('reply', ascending=False)

print(f"Tweet dengan reply > 5: {len(df_kandidat)} tweet")
print()
print("Tweet-tweet terpopuler (berdasarkan jumlah reply):")
df_kandidat[['tweet_id', 'username', 'teks', 'reply', 'like']].head(10)

In [ ]:
# Langkah 2: Pilih 5 tweet teratas untuk diambil replies-nya
tweet_ids_target = df_kandidat['tweet_id'].head(5).tolist()

print("Tweet yang akan diambil replies-nya:")
for tid in tweet_ids_target:
    baris = df_kandidat[df_kandidat['tweet_id'] == tid].iloc[0]
    print(f"  [{tid}] @{baris['username']} -- '{baris['teks'][:60]}...' ({baris['reply']} replies)")

In [ ]:
# ==============================================================
#  FUNGSI UTAMA: Ambil replies dari beberapa tweet
# ==============================================================

def ambil_replies(tweet_ids, maks_reply_per_tweet=50, simpan_ke=None):
    """
    Mengambil replies dari beberapa tweet sekaligus menggunakan TwitterAPI.io.

    Parameter:
        tweet_ids           : list berisi ID tweet yang ingin diambil replies-nya
        maks_reply_per_tweet: maksimum reply yang diambil per tweet
        simpan_ke           : (opsional) path file JSON untuk menyimpan hasil

    Return:
        list of dict -- semua reply dari semua tweet
    """
    semua_replies = []

    for i, tweet_id in enumerate(tweet_ids, 1):
        print(f"  [{i}/{len(tweet_ids)}] Mengambil replies dari tweet {tweet_id}...")

        replies_tweet_ini = []
        ids_seen_reply    = set()
        cursor            = None
        cursor_lama       = None

        while len(replies_tweet_ini) < maks_reply_per_tweet:
            params = {"tweetId": str(tweet_id)}
            if cursor:
                params["cursor"] = cursor

            try:
                resp = requests.get(
                    f"{BASE_URL}/twitter/tweet/replies",
                    headers=HEADERS, params=params, timeout=15
                )

                if resp.status_code != 200:
                    print(f"     Error {resp.status_code}")
                    break

                data = resp.json()
                replies_halaman = data.get("tweets", [])

                if not replies_halaman:
                    break

                baru = [r for r in replies_halaman if r.get('id') not in ids_seen_reply]
                if not baru:
                    break
                for r in baru:
                    ids_seen_reply.add(r.get('id'))
                    r['_parent_tweet_id'] = str(tweet_id)
                replies_tweet_ini.extend(baru)

                cursor_baru = data.get("next_cursor")
                if not cursor_baru or cursor_baru == cursor_lama:
                    break
                cursor_lama = cursor
                cursor      = cursor_baru

                time.sleep(0.5)
            except Exception as e:
                print(f"     Gagal: {e}")
                break

        replies_tweet_ini = replies_tweet_ini[:maks_reply_per_tweet]
        semua_replies.extend(replies_tweet_ini)
        print(f"     {len(replies_tweet_ini)} reply ditemukan")
        time.sleep(5)  # jeda antar tweet

    print(f"\nTotal: {len(semua_replies)} reply dari {len(tweet_ids)} tweet.")

    if simpan_ke:
        with open(simpan_ke, 'w', encoding='utf-8') as f:
            json.dump(semua_replies, f, ensure_ascii=False, indent=2)
        print(f"Data disimpan ke: {simpan_ke}")

    return semua_replies

print("Fungsi ambil_replies() siap digunakan.")

In [ ]:
# Jalankan pengambilan replies dari 5 tweet terpopuler
print("Mengambil replies...")
print("=" * 50)

semua_replies = ambil_replies(
    tweet_ids=tweet_ids_target,
    maks_reply_per_tweet=50,
    simpan_ke="replies_hasil.json"
)

### 4.1 Membangun Network dari Replies

Sekarang kita punya data reply. Saatnya mengubahnya menjadi sebuah **directed weighted network** menggunakan NetworkX -- persis seperti yang sudah kita pelajari di Lab 1.

In [ ]:
def bangun_reply_network(replies, tweet_asal):
    """
    Membangun directed weighted network dari data replies.

    Logika:
    - Setiap reply adalah edge: replier -----> penulis tweet yang dibalas
    - Jika A membalas B beberapa kali, bobot edge-nya bertambah

    Parameter:
        replies    : list of dict hasil ambil_replies()
        tweet_asal : DataFrame tweet asli (untuk mendapatkan username penulis)

    Return:
        nx.DiGraph -- network percakapan
    """
    # Mapping: tweet_id -> username penulis tweet asal
    id_ke_user = dict(zip(
        tweet_asal['tweet_id'].astype(str),
        tweet_asal['username']
    ))

    G = nx.DiGraph()
    ditambahkan = 0
    diabaikan   = 0

    for reply in replies:
        # Siapa yang membalas?
        replier = reply.get('author', {}).get('userName')

        # Siapa yang dibalas?
        # TwitterAPI.io menyediakan field inReplyToUsername secara langsung
        in_reply_to_user = reply.get('inReplyToUsername')

        # Fallback: cari dari mapping tweet_id -> username
        if not in_reply_to_user:
            in_reply_to_id   = str(reply.get('inReplyToId', ''))
            in_reply_to_user = id_ke_user.get(in_reply_to_id)

        # Buat edge jika keduanya valid dan bukan self-loop
        if replier and in_reply_to_user and replier != in_reply_to_user:
            if G.has_edge(replier, in_reply_to_user):
                G[replier][in_reply_to_user]['weight'] += 1
            else:
                G.add_edge(replier, in_reply_to_user, weight=1)
            ditambahkan += 1
        else:
            diabaikan += 1

    print("Network berhasil dibangun.")
    print(f"  Node (pengguna)  : {G.number_of_nodes()}")
    print(f"  Edge (interaksi) : {G.number_of_edges()}")
    print(f"  Edge ditambahkan : {ditambahkan}")
    print(f"  Edge diabaikan   : {diabaikan} (data tidak lengkap)")
    return G

G_percakapan = bangun_reply_network(semua_replies, df)

### 4.2 Analisis Network Percakapan

Network sudah terbentuk. Sekarang kita terapkan analisis yang sudah dipelajari di Lab 3-4.

In [ ]:
print("STATISTIK NETWORK PERCAKAPAN")
print("=" * 50)
print(f"Jumlah node (pengguna aktif) : {G_percakapan.number_of_nodes()}")
print(f"Jumlah edge (interaksi unik) : {G_percakapan.number_of_edges()}")

density = nx.density(G_percakapan)
print(f"Density network              : {density:.4f}")
print(f"  -> Hanya {density*100:.2f}% dari semua kemungkinan interaksi yang benar-benar terjadi")

resiprositas = nx.reciprocity(G_percakapan)
print(f"Resiprositas                 : {resiprositas:.2%}")
print(f"  -> {resiprositas:.2%} interaksi bersifat dua arah (saling balas)")

G_undir      = G_percakapan.to_undirected()
n_komponen   = nx.number_connected_components(G_undir)
komp_terbesar = max(nx.connected_components(G_undir), key=len)
print(f"Jumlah komponen              : {n_komponen}")
print(f"Ukuran komponen terbesar     : {len(komp_terbesar)} node ({len(komp_terbesar)/G_percakapan.number_of_nodes()*100:.1f}%)")

In [ ]:
in_degree  = dict(G_percakapan.in_degree())
out_degree = dict(G_percakapan.out_degree())

print("TOP 10 PENGGUNA BERDASARKAN IN-DEGREE (paling sering dibalas):")
print("-" * 55)
for rank, (user, deg) in enumerate(sorted(in_degree.items(), key=lambda x: -x[1])[:10], 1):
    print(f"  {rank:2}. @{user:<20} {deg} kali dibalas")

print()
print("TOP 5 PENGGUNA BERDASARKAN OUT-DEGREE (paling aktif membalas):")
print("-" * 55)
for rank, (user, deg) in enumerate(sorted(out_degree.items(), key=lambda x: -x[1])[:5], 1):
    print(f"  {rank:2}. @{user:<20} membalas {deg} pengguna berbeda")

In [ ]:
# Visualisasi network -- fokus pada komponen terbesar
G_sub = G_percakapan.subgraph(komp_terbesar).copy()

in_deg      = dict(G_sub.in_degree())
ukuran_node = [in_deg[n] * 80 + 100 for n in G_sub.nodes()]
top5_users  = {u for u, _ in sorted(in_deg.items(), key=lambda x: -x[1])[:5]}
warna_node  = ['#F59E0B' if n in top5_users else '#065A82' for n in G_sub.nodes()]

bobot_edge  = [G_sub[u][v]['weight'] for u, v in G_sub.edges()]
maks_bobot  = max(bobot_edge) if bobot_edge else 1
lebar_edge  = [w / maks_bobot * 3 + 0.3 for w in bobot_edge]

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G_sub, seed=42, k=0.8)

nx.draw(G_sub, pos,
        node_color=warna_node,
        node_size=ukuran_node,
        edge_color='#cccccc',
        width=lebar_edge,
        with_labels=False,
        alpha=0.85,
        arrows=True,
        arrowsize=8,
        connectionstyle='arc3,rad=0.05')

label_penting = {n: f'@{n}' for n in top5_users if n in G_sub.nodes()}
nx.draw_networkx_labels(G_sub, pos, labels=label_penting,
                        font_size=9, font_color='white', font_weight='bold')

legenda = [
    mpatches.Patch(color='#F59E0B', label='Top 5 in-degree tertinggi'),
    mpatches.Patch(color='#065A82', label='Pengguna lain'),
]
plt.legend(handles=legenda, loc='upper left', fontsize=10)
plt.title(
    f"Conversation Network: Topik 'Iran' di Twitter\n"
    f"({G_sub.number_of_nodes()} pengguna, {G_sub.number_of_edges()} interaksi)",
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("Node kuning = pengguna yang paling sering dibalas (hub percakapan)")
print("Ukuran node = in-degree")
print("Ketebalan garis = bobot interaksi")

### [Pertanyaan 2]

Perhatikan visualisasi conversation network di atas.

1. Apakah struktur network ini lebih mirip model **Erdos-Renyi**, **Watts-Strogatz**, atau **Barabasi-Albert** dari Lab sebelumnya? Jelaskan berdasarkan apa yang Anda lihat.
2. Node yang paling banyak dibalas (hub) -- dalam konteks diskusi Iran di Twitter, tipe pengguna apa yang kemungkinan besar menjadi hub? (misalnya: jurnalis, pejabat, akun berita?)
3. Nilai resiprositas yang dihitung sebelumnya -- apakah tinggi atau rendah? Apa artinya tentang sifat percakapan di Twitter?

**Jawaban Anda:**

1. ...
2. ...
3. ...

---
## 5. Menghubungkan ke Feature Extraction (Lab 3)

Di Lab 3, kita sudah belajar mengekstrak **fitur network** dari sebuah graph. Sekarang kita terapkan hal yang sama pada network percakapan nyata -- hasilnya bisa langsung digunakan sebagai fitur untuk machine learning.

In [ ]:
print("Menghitung metrik sentralitas...")

in_deg_centrality  = nx.in_degree_centrality(G_percakapan)
out_deg_centrality = nx.out_degree_centrality(G_percakapan)
pagerank           = nx.pagerank(G_percakapan, weight='weight')

# Betweenness dihitung pada komponen terbesar karena graph tidak fully connected
G_sub_dir   = G_percakapan.subgraph(komp_terbesar)
betweenness = nx.betweenness_centrality(G_sub_dir, weight='weight', normalized=True)

df_fitur = pd.DataFrame({
    'username'         : list(G_percakapan.nodes()),
    'in_degree'        : [in_degree.get(n, 0)         for n in G_percakapan.nodes()],
    'out_degree'       : [out_degree.get(n, 0)        for n in G_percakapan.nodes()],
    'in_deg_centrality': [in_deg_centrality.get(n, 0) for n in G_percakapan.nodes()],
    'pagerank'         : [pagerank.get(n, 0)          for n in G_percakapan.nodes()],
    'betweenness'      : [betweenness.get(n, 0)       for n in G_percakapan.nodes()],
}).sort_values('pagerank', ascending=False)

print("Fitur network berhasil dihitung.")
print("\n10 pengguna paling berpengaruh berdasarkan PageRank:")
df_fitur.head(10).round(4)

In [ ]:
top15 = df_fitur.head(15).copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrik_info = [
    ('in_degree',  '#065A82', 'In-Degree\n(Paling sering dibalas)'),
    ('pagerank',   '#E74C3C', 'PageRank\n(Pengaruh mempertimbangkan kualitas koneksi)'),
    ('betweenness','#2ECC71', 'Betweenness\n(Penghubung antar kelompok)'),
]

for ax, (kolom, warna, judul) in zip(axes, metrik_info):
    data_plot = top15.dropna(subset=[kolom]).nlargest(10, kolom)
    ax.barh(data_plot['username'][::-1], data_plot[kolom][::-1], color=warna, alpha=0.85)
    ax.set_title(judul, fontsize=11, fontweight='bold')
    ax.set_xlabel('Nilai', fontsize=10)

plt.suptitle("Perbandingan Metrik Sentralitas pada Conversation Network",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Perhatikan: ranking pengguna bisa berbeda tergantung metrik yang digunakan.")
print("  In-degree   = siapa yang paling sering dibalas")
print("  PageRank    = siapa yang paling berpengaruh")
print("  Betweenness = siapa yang menjembatani kelompok berbeda")

In [ ]:
# Simpan fitur ke CSV -- siap digunakan sebagai input machine learning
df_fitur.to_csv("fitur_network_percakapan.csv", index=False)
print("Fitur network disimpan ke 'fitur_network_percakapan.csv'")
print("File ini siap digunakan sebagai input model ML di lab-lab berikutnya.")

### [Pertanyaan 3]

Lihat perbandingan ketiga metrik sentralitas di atas.

1. Apakah pengguna dengan **in-degree tertinggi** selalu sama dengan yang **PageRank tertinggi**? Mengapa bisa berbeda?
2. Pengguna dengan **betweenness tinggi** disebut sebagai *broker* atau *jembatan* dalam network. Dalam konteks diskusi banjir di Twitter, apa peran sosial pengguna seperti ini?
3. Fitur-fitur sentralitas ini bisa digunakan untuk apa dalam machine learning? Berikan satu contoh task klasifikasi yang bisa memanfaatkan fitur ini.

**Jawaban Anda:**

1. ...
2. ...
3. ...

---
## 6. Tugas: Analisis Topik Pilihan

### Pilih Topik

Pilih **salah satu** topik diskusi yang sedang ramai:
- Isu lingkungan (banjir, kebakaran hutan, polusi udara)
- Isu sosial (pendidikan, kesehatan, lapangan kerja)
- Isu politik
- Topik olahraga (sepak bola, bulutangkis)
- Bebas, asalkan ada cukup tweet dan diskusi aktif

### Langkah-langkah

**A. Pengumpulan Data**
1. Rancang minimal **2 query berbeda** untuk topik yang Anda pilih (bisa gunakan advanced search syntax)
2. Kumpulkan minimal **500 tweet** total
3. Pilih minimal **5 tweet** dengan reply terbanyak untuk diambil replies-nya
4. Simpan semua data ke file JSON

**B. Analisis Network**
1. Bangun conversation network dari data replies
2. Visualisasikan network dengan ukuran node berdasarkan **in-degree** dan warna berdasarkan **betweenness** (gunakan colormap)
3. Hitung dan bandingkan ketiga metrik sentralitas

**C. Interpretasi**
1. Siapa yang paling berpengaruh di diskusi ini (berdasarkan PageRank)? Siapa mereka?
2. Apakah ada perbedaan antara pengguna dengan in-degree tinggi vs betweenness tinggi?
3. Apakah struktur network ini mirip dengan salah satu model yang dipelajari di Lab 1? Dukung dengan data.
4. Fitur network apa saja yang bisa Anda ekstrak, dan berikan contoh task ML yang bisa menggunakannya?

In [ ]:

# === TUGAS A: Pengumpulan Data ===

TOPIK = "..."  # isi dengan topik pilihan kelompok

# Query 1
# tweets_q1 = cari_tweet(query="...", maks_tweet=300, simpan_ke="tugas_q1.json")

# Query 2
# tweets_q2 = cari_tweet(query="...", maks_tweet=300, simpan_ke="tugas_q2.json")

# Gabungkan semua tweet
# tweets_semua = tweets_q1 + tweets_q2
# df_tugas = tweets_ke_dataframe(tweets_semua)
# print(f"Total tweet: {len(df_tugas)}")

In [ ]:
# === TUGAS B: Analisis Network ===

# ids_tugas = df_tugas[df_tugas['reply'] > 5].sort_values('reply', ascending=False)['tweet_id'].head(5).tolist()
# replies_tugas = ambil_replies(ids_tugas, maks_reply_per_tweet=50, simpan_ke="tugas_replies.json")
# G_tugas = bangun_reply_network(replies_tugas, df_tugas)

# Visualisasi dengan colormap untuk betweenness
# betweenness_tugas = nx.betweenness_centrality(G_tugas, weight='weight', normalized=True)
# warna_betweenness = [betweenness_tugas.get(n, 0) for n in G_tugas.nodes()]
#
# plt.figure(figsize=(14, 10))
# pos_tugas    = nx.spring_layout(G_tugas, seed=42)
# in_deg_tugas = dict(G_tugas.in_degree())
# ukuran_tugas = [in_deg_tugas[n] * 100 + 100 for n in G_tugas.nodes()]
#
# nx.draw(G_tugas, pos_tugas,
#         node_color=warna_betweenness, node_size=ukuran_tugas,
#         cmap=plt.cm.YlOrRd, edge_color='#cccccc', width=0.5,
#         with_labels=False, arrows=True)
# plt.colorbar(plt.cm.ScalarMappable(cmap=plt.cm.YlOrRd), label='Betweenness Centrality')
# plt.title(f"Conversation Network: {TOPIK}")
# plt.show()

In [ ]:
# === TUGAS C: Analisis Sentralitas ===

# Hitung semua metrik dan buat tabel perbandingan
# ...

# Simpan fitur ke CSV
# df_fitur_tugas.to_csv("fitur_tugas.csv", index=False)
# print("Fitur disimpan.")

### Analisis Tertulis

Tulis analisis kelompok Anda di sini:

**Topik yang dipilih:** ...

**Query yang digunakan:**
- Query 1: `...` -- alasan pemilihan: ...
- Query 2: `...` -- alasan pemilihan: ...

**C1. Pengguna paling berpengaruh:**
- Berdasarkan PageRank: @... (siapa mereka? akun apa?)
- Apakah berbeda dengan in-degree tertinggi? Mengapa?

**C2. In-degree vs Betweenness:**
- ...

**C3. Perbandingan dengan model network:**
- Network ini paling mirip model ... karena ...
- Bukti kuantitatif: ...

**C4. Potensi penggunaan untuk data science/machine learning:**
- Fitur yang dapat diekstrak: ...
- Contoh task: ...

---

### Referensi

- [Dokumentasi TwitterAPI.io](https://docs.twitterapi.io)
- [Twitter Advanced Search Syntax](https://developer.twitter.com/en/docs/twitter-api/tweets/search/integrate/build-a-query)
- [Dokumentasi NetworkX](https://networkx.org/documentation/stable/)
- Barabasi, A. L. (2016). *Network Science*. [networksciencebook.com](http://networksciencebook.com)